# 03. Preparación del corpus para recuperación de evidencia — ONPE 2026

## Propósito

Este notebook continúa el trabajo de los notebooks `01_exploracion_datos.ipynb` y `02_validacion_unidades_textuales.ipynb`.

- El notebook 01 explora los JSON descargados.
- El notebook 02 valida las unidades textuales generadas desde esos JSON.
- Este notebook prepara representaciones alternativas de las **unidades textuales existentes** para usarlas más adelante en experimentos de recuperación de evidencia.

**No vuelve a convertir los JSON a TXT**, no descarga datos, no ejecuta recuperadores BM25/semánticos/híbridos y no calcula métricas de recuperación. No se declara una variante ganadora.

### Variantes que se preparan

- **A — Base:** contenido de la unidad textual existente, conservado tal como está.
- **B — Normalización conservadora:** mismos datos y orden, normalizando espacios y finales de línea.
- **C — Contexto de ruta:** texto normalizado más metadatos derivados de la ruta y del nombre de archivo. No se inventan nombres de departamentos, provincias ni distritos a partir de códigos.

> Antes de interpretar los resultados, revisa las auditorías y asegúrate de que las unidades textuales estén alineadas con sus JSON de origen. Una ruta o nombre de archivo no demuestra por sí solo la cobertura geográfica.

## 1. Configuración y localización del proyecto

El notebook busca la estructura existente `data/processed/unidades_textuales` y `data/raw/ONPE_2026/Peru`. Si no la encuentra, ajusta `RAIZ` manualmente.

In [ ]:
from pathlib import Path
import json
import re
import hashlib
import pandas as pd
from datetime import datetime

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 60)

def encontrar_raiz(inicio: Path) -> Path:
    inicio = inicio.resolve()
    for candidato in [inicio, *inicio.parents]:
        if (candidato / "data" / "processed" / "unidades_textuales").exists():
            return candidato
        if (candidato / "data" / "raw" / "ONPE_2026" / "Peru").exists():
            return candidato
        if (candidato / "README.md").exists() and (candidato / "notebooks").exists():
            return candidato
    return inicio.parent if inicio.name == "notebooks" else inicio

RAIZ = encontrar_raiz(Path.cwd())
RUTA_JSON = RAIZ / "data" / "raw" / "ONPE_2026" / "Peru"
RUTA_TEXTOS = RAIZ / "data" / "processed" / "unidades_textuales"
RUTA_SALIDA = RAIZ / "data" / "processed" / "feature_engineering"
RUTA_RESULTADOS = RAIZ / "results" / "feature_engineering"
RUTA_EVALUACION = RAIZ / "data" / "evaluation"

for carpeta in [RUTA_SALIDA, RUTA_RESULTADOS, RUTA_EVALUACION]:
    carpeta.mkdir(parents=True, exist_ok=True)

print("Directorio actual:", Path.cwd().resolve())
print("Raíz detectada:", RAIZ)
print("JSON:", RUTA_JSON, "| existe:", RUTA_JSON.exists())
print("Unidades textuales:", RUTA_TEXTOS, "| existe:", RUTA_TEXTOS.exists())

## 2. Inventario de unidades textuales existentes

La entrada principal son los `.txt` que ya generaste y validaste parcialmente en el notebook 02.

In [ ]:
if not RUTA_TEXTOS.exists():
    raise FileNotFoundError(
        f"No se encuentra {RUTA_TEXTOS}. Ejecuta primero el generador de unidades textuales "
        "o corrige la ruta RUTA_TEXTOS."
    )

archivos_txt = sorted(p for p in RUTA_TEXTOS.rglob("*.txt") if p.is_file())
print(f"Unidades textuales encontradas: {len(archivos_txt):,}")

filas = []
for p in archivos_txt:
    try:
        texto = p.read_text(encoding="utf-8-sig")
        rel = p.relative_to(RUTA_TEXTOS).as_posix()
        filas.append({
            "source_path": rel,
            "file_name": p.name,
            "bytes": p.stat().st_size,
            "chars": len(texto),
            "lines": len(texto.splitlines()),
            "empty": not bool(texto.strip()),
            "read_error": ""
        })
    except Exception as exc:
        filas.append({
            "source_path": str(p),
            "file_name": p.name,
            "bytes": p.stat().st_size,
            "chars": None,
            "lines": None,
            "empty": None,
            "read_error": f"{type(exc).__name__}: {exc}"
        })

df_inventario = pd.DataFrame(filas)
display(df_inventario.head(15))
df_inventario.to_csv(RUTA_RESULTADOS / "inventario_unidades_feature_engineering.csv",
                     index=False, encoding="utf-8-sig")

## 3. Auditoría de metadatos y rastreo hacia el JSON de origen

Se comprueba si cada unidad tiene un marcador `FUENTE:` o `ARCHIVO_ORIGEN:` y si puede localizarse el JSON correspondiente. Esta comprobación es orientativa; revisa los casos fallidos manualmente.

In [ ]:
def resolver_json(archivo_txt: Path):
    relativa = archivo_txt.relative_to(RUTA_TEXTOS)
    nombre_base = relativa.name
    if nombre_base.endswith("_unidad_textual.txt"):
        nombre_base = nombre_base[:-len("_unidad_textual.txt")]
    else:
        nombre_base = relativa.stem
    candidato_rel = relativa.with_name(nombre_base + ".json")
    candidato = RUTA_JSON / candidato_rel
    if candidato.exists():
        return candidato

    partes = list(candidato_rel.parts)
    if partes and partes[0].lower() == "peru":
        candidato2 = RUTA_JSON.joinpath(*partes[1:])
        if candidato2.exists():
            return candidato2

    # Respaldo: leer marcadores de rastreo de la unidad textual.
    try:
        lineas = archivo_txt.read_text(encoding="utf-8-sig").splitlines()
    except Exception:
        return None
    metadatos = {}
    for linea in lineas:
        if ":" in linea:
            k, v = linea.split(":", 1)
            if k.strip() in {"FUENTE", "ARCHIVO_ORIGEN"}:
                metadatos[k.strip()] = v.strip()
    origen = metadatos.get("ARCHIVO_ORIGEN") or metadatos.get("FUENTE")
    if origen:
        p = Path(origen)
        opciones = [p] if p.is_absolute() else [RAIZ / p, RUTA_JSON / p]
        for opcion in opciones:
            if opcion.exists() and opcion.is_file() and opcion.suffix.lower() == ".json":
                return opcion
    return None

auditoria = []
for p in archivos_txt:
    try:
        texto = p.read_text(encoding="utf-8-sig")
        lineas = texto.splitlines()
        fuente = next((x.split(":", 1)[1].strip() for x in lineas if x.startswith("FUENTE:")), "")
        origen = next((x.split(":", 1)[1].strip() for x in lineas if x.startswith("ARCHIVO_ORIGEN:")), "")
        json_origen = resolver_json(p)
        auditoria.append({
            "unidad": p.relative_to(RUTA_TEXTOS).as_posix(),
            "tiene_fuente": bool(fuente),
            "tiene_archivo_origen": bool(origen),
            "tiene_metadato_rastreo": bool(fuente or origen),
            "json_origen_localizado": json_origen is not None,
            "ruta_json_resuelta": str(json_origen.relative_to(RAIZ)) if json_origen and json_origen.is_relative_to(RAIZ) else (str(json_origen) if json_origen else ""),
            "texto_no_vacio": bool(texto.strip()),
            "error": ""
        })
    except Exception as exc:
        auditoria.append({
            "unidad": str(p), "tiene_fuente": False, "tiene_archivo_origen": False,
            "tiene_metadato_rastreo": False, "json_origen_localizado": False,
            "ruta_json_resuelta": "", "texto_no_vacio": False,
            "error": f"{type(exc).__name__}: {exc}"
        })

df_auditoria = pd.DataFrame(auditoria)
display(pd.DataFrame({
    "control": ["Tiene FUENTE", "Tiene ARCHIVO_ORIGEN", "Tiene algún metadato",
                "JSON de origen localizado", "Texto no vacío"],
    "cantidad": [
        int(df_auditoria["tiene_fuente"].sum()),
        int(df_auditoria["tiene_archivo_origen"].sum()),
        int(df_auditoria["tiene_metadato_rastreo"].sum()),
        int(df_auditoria["json_origen_localizado"].sum()),
        int(df_auditoria["texto_no_vacio"].sum())
    ],
    "total": [len(df_auditoria)] * 5
}))
display(df_auditoria[
    (~df_auditoria["tiene_metadato_rastreo"]) |
    (~df_auditoria["json_origen_localizado"]) |
    (~df_auditoria["texto_no_vacio"]) |
    (df_auditoria["error"] != "")
].head(50))
df_auditoria.to_csv(RUTA_RESULTADOS / "auditoria_rastreo_feature_engineering.csv",
                    index=False, encoding="utf-8-sig")

## 4. Construcción de variantes A, B y C

La transformación se aplica a los textos existentes. La variante B no elimina tildes, no cambia cifras ni reordena campos. La variante C agrega información explícita de archivo y ruta como contexto, pero no traduce códigos UBIGEO a nombres geográficos.

In [ ]:
def normalizar_espacios(texto: str) -> str:
    # Conserva líneas y valores, pero unifica espacios repetidos y finales de línea.
    lineas = texto.replace("\r\n", "\n").replace("\r", "\n").split("\n")
    lineas = [re.sub(r"[ \t]+", " ", linea).strip() for linea in lineas]
    # Colapsa líneas vacías consecutivas sin alterar el orden de los datos.
    salida = []
    vacia_previa = False
    for linea in lineas:
        vacia = not linea
        if vacia and vacia_previa:
            continue
        salida.append(linea)
        vacia_previa = vacia
    return "\n".join(salida).strip()

documentos = []
for p in archivos_txt:
    try:
        texto_a = p.read_text(encoding="utf-8-sig")
    except Exception:
        continue

    rel = p.relative_to(RUTA_TEXTOS).as_posix()
    texto_b = normalizar_espacios(texto_a)
    ruta_contexto = " / ".join(Path(rel).parts[:-1])
    texto_c = (
        f"ARCHIVO: {p.name}\n"
        f"RUTA_RELATIVA: {rel}\n"
        f"CONTEXTO_CARPETAS: {ruta_contexto or '(raíz de unidades textuales)'}\n\n"
        f"{texto_b}"
    ).strip()

    doc_id = hashlib.sha256(rel.encode("utf-8")).hexdigest()[:16]
    documentos.append({
        "doc_id": doc_id,
        "source_path": rel,
        "file_name": p.name,
        "sha256_original": hashlib.sha256(texto_a.encode("utf-8")).hexdigest(),
        "chars_A": len(texto_a),
        "chars_B": len(texto_b),
        "chars_C": len(texto_c),
        "text_A": texto_a,
        "text_B": texto_b,
        "text_C": texto_c
    })

df_docs = pd.DataFrame(documentos)
print(f"Documentos preparados: {len(df_docs):,}")
display(df_docs[["doc_id", "source_path", "chars_A", "chars_B", "chars_C"]].head(10))

## 5. Exportación de los corpus en formato JSONL

Cada línea contiene un documento y sus metadatos de trazabilidad. Estos archivos son entradas para futuros experimentos; su creación no significa que la recuperación ya haya sido evaluada.

In [ ]:
def exportar_jsonl(df, columna_texto, destino):
    with destino.open("w", encoding="utf-8") as f:
        for _, fila in df.iterrows():
            registro = {
                "doc_id": fila["doc_id"],
                "source_path": fila["source_path"],
                "file_name": fila["file_name"],
                "text": fila[columna_texto],
                "sha256_original": fila["sha256_original"]
            }
            f.write(json.dumps(registro, ensure_ascii=False) + "\n")

exportar_jsonl(df_docs, "text_A", RUTA_SALIDA / "corpus_A_baseline.jsonl")
exportar_jsonl(df_docs, "text_B", RUTA_SALIDA / "corpus_B_normalizado.jsonl")
exportar_jsonl(df_docs, "text_C", RUTA_SALIDA / "corpus_C_contexto_ruta.jsonl")

df_catalogo = df_docs[[
    "doc_id", "source_path", "file_name", "sha256_original",
    "chars_A", "chars_B", "chars_C"
]].copy()
df_catalogo.to_csv(RUTA_SALIDA / "catalogo_evidencias.csv",
                   index=False, encoding="utf-8-sig")
print("Corpus exportados en:", RUTA_SALIDA)

## 6. Resumen descriptivo de las variantes

Estas estadísticas describen el tamaño de los textos. No son métricas de recuperación y no permiten afirmar que una variante sea mejor.

In [ ]:
resumen = []
for nombre, col in [
    ("A_baseline", "text_A"),
    ("B_normalizado", "text_B"),
    ("C_contexto_ruta", "text_C")
]:
    longitudes = df_docs[col].fillna("").str.len()
    resumen.append({
        "variante": nombre,
        "documentos": len(df_docs),
        "caracteres_totales": int(longitudes.sum()),
        "media_caracteres": round(float(longitudes.mean()), 2) if len(longitudes) else 0,
        "mediana_caracteres": float(longitudes.median()) if len(longitudes) else 0,
        "documentos_vacios": int((longitudes == 0).sum())
    })
df_resumen = pd.DataFrame(resumen)
display(df_resumen)
df_resumen.to_csv(RUTA_RESULTADOS / "resumen_variantes.csv",
                  index=False, encoding="utf-8-sig")

## 7. Plantilla de afirmaciones para evaluación

La plantilla queda vacía intencionalmente. Las afirmaciones deben construirse y revisarse por separado del corpus de documentos. `reference_doc_ids` debe contener los IDs de las evidencias que permiten evaluar cada afirmación; no se deben inventar etiquetas.

In [ ]:
RUTA_CLAIMS = RUTA_EVALUACION / "claims_evaluacion_template.csv"
if not RUTA_CLAIMS.exists():
    pd.DataFrame(columns=[
        "claim_id", "claim_text", "claim_type", "geographic_scope",
        "geographic_code", "reference_doc_ids", "gold_label",
        "annotation_notes", "annotator", "review_status"
    ]).to_csv(RUTA_CLAIMS, index=False, encoding="utf-8-sig")
print("Plantilla de evaluación:", RUTA_CLAIMS)
display(pd.read_csv(RUTA_CLAIMS, encoding="utf-8-sig").head())

## 8. Registro de ejecución

Se registra cuántas unidades se procesaron y dónde se guardaron los resultados. No se registran Recall@k, MRR ni latencia porque este notebook todavía no ejecuta búsquedas.

In [ ]:
registro = pd.DataFrame([{
    "fecha_ejecucion": datetime.now().isoformat(timespec="seconds"),
    "raiz_proyecto": str(RAIZ),
    "ruta_unidades_textuales": str(RUTA_TEXTOS),
    "unidades_txt_encontradas": len(archivos_txt),
    "documentos_procesados": len(df_docs),
    "corpus_A": str(RUTA_SALIDA / "corpus_A_baseline.jsonl"),
    "corpus_B": str(RUTA_SALIDA / "corpus_B_normalizado.jsonl"),
    "corpus_C": str(RUTA_SALIDA / "corpus_C_contexto_ruta.jsonl")
}])
log_path = RUTA_RESULTADOS / "registro_feature_engineering.csv"
if log_path.exists():
    anterior = pd.read_csv(log_path, encoding="utf-8-sig")
    registro = pd.concat([anterior, registro], ignore_index=True)
registro.to_csv(log_path, index=False, encoding="utf-8-sig")
display(registro.tail(1))

## 9. Controles antes de pasar a recuperación

- [ ] El notebook 02 terminó sin errores críticos.
- [ ] Revisé los casos de la auditoría donde no se localiza el JSON de origen.
- [ ] Verifiqué manualmente ejemplos de A, B y C.
- [ ] Confirmé que B conserva nombres, tildes y valores numéricos.
- [ ] Confirmé que C solo agrega metadatos derivados de la ruta y el archivo.
- [ ] Revisé la correspondencia entre archivos y ámbito geográfico; no asumí que el nombre del archivo prueba cobertura nacional.
- [ ] Construí afirmaciones de evaluación y asigné evidencia de referencia de forma independiente.
- [ ] Mantuve las afirmaciones fuera del corpus usado por el recuperador.

### Salidas principales

- `data/processed/feature_engineering/corpus_A_baseline.jsonl`
- `data/processed/feature_engineering/corpus_B_normalizado.jsonl`
- `data/processed/feature_engineering/corpus_C_contexto_ruta.jsonl`
- `data/processed/feature_engineering/catalogo_evidencias.csv`
- `data/evaluation/claims_evaluacion_template.csv`
- `results/feature_engineering/auditoria_rastreo_feature_engineering.csv`
- `results/feature_engineering/resumen_variantes.csv`
- `results/feature_engineering/registro_feature_engineering.csv`